# NYC Taxi Trip Duration Modeling
## Leakage-Aware Regression on Large-Scale Urban Mobility Data
### Supplementary Analysis of Payment and Tipping Behavior


## 1. Introduction / Background

New York City taxi trip data provides a rich real-world example of structured transportation data. The primary goal of this project is to predict trip duration using features that are available before or at the beginning of a trip. This is formulated as a regression problem.

The project focuses on building a realistic predictive pipeline by avoiding data leakage from post-trip monetary variables such as fare amount, tip amount, tolls, surcharges, and total amount. These variables are useful for analysis, but they are not available at trip start and therefore should not be used for ETA-style duration prediction.

In addition to prediction, the project includes a supplementary hypothesis-testing component to examine whether average tip rates differ across comparable payment categories and time groups.


## 2. Dataset Description

This project uses the **January 2026 NYC Yellow Taxi Trip Records** published by the NYC Taxi and Limousine Commission (TLC). The dataset is public, structured, and trip-level, making it suitable for large-scale tabular modeling.

Before cleaning or modeling, the notebook inspects the raw dataset structure, including size, column names, data types, and missingness patterns. This helps identify which fields require transformation and which variables should be excluded from the predictive task to prevent leakage.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

data_path = PROJECT_ROOT / "yellow_tripdata_2026-01.parquet"

df = pd.read_parquet(data_path)

print("Raw shape:", df.shape)
display(df.head())

print("Column names:")
print(df.columns.tolist())

print("Data types:")
display(df.dtypes)

missing_summary = df.isna().sum().sort_values(ascending=False)
print("Missing values (nonzero only):")
display(missing_summary[missing_summary > 0])

## 3. Data Cleaning and Preprocessing

### 3.1 Feature creation

The raw taxi dataset contains timestamp fields that must be converted into datetime format before we can derive time-related features.
We create several variables needed for downstream analysis:

- `trip_duration_min`: the main regression target
- `pickup_hour`: hour of day
- `pickup_dayofweek`: day of week
- `is_weekend`: weekend indicator
- `tip_rate`: tip amount divided by fare amount

Among these variables, `trip_duration_min` is the primary target of the project, while `tip_rate` is reserved only for the supplementary tipping-analysis component.


In [ ]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['trip_duration_min'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60

df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['tpep_pickup_datetime'].dt.dayofweek
df['is_weekend'] = df['pickup_dayofweek'].isin([5, 6]).astype(int)

df['tip_rate'] = np.where(
    df['fare_amount'] > 0,
    df['tip_amount'] / df['fare_amount'],
    np.nan
)

display(df[[
    'trip_duration_min',
    'pickup_hour',
    'pickup_dayofweek',
    'is_weekend',
    'tip_rate'
]].head())


In [ ]:
feature_notes = pd.DataFrame({
    'feature': [
        'trip_duration_min',
        'pickup_hour',
        'pickup_dayofweek',
        'is_weekend',
        'tip_rate'
    ],
    'purpose': [
        'Main regression target',
        'Time-of-day feature',
        'Day-of-week feature',
        'Weekend indicator',
        'Supplementary tipping analysis only'
    ]
})
display(feature_notes)


### 3.2 Base cleaning rules

We next apply a conservative set of base-cleaning rules to remove clearly invalid trip records.
Specifically, we keep only observations with:

- positive trip duration
- positive trip distance
- positive fare amount
- positive total amount

At this stage, the goal is not to aggressively remove unusual trips, but rather to eliminate records that are clearly invalid or unsuitable for downstream analysis.


In [ ]:
invalid_duration = (df['trip_duration_min'] <= 0).sum()
invalid_distance = (df['trip_distance'] <= 0).sum()
invalid_fare = (df['fare_amount'] <= 0).sum()
invalid_total = (df['total_amount'] <= 0).sum()

cleaning_audit = pd.DataFrame({
    'rule': [
        'trip_duration_min <= 0',
        'trip_distance <= 0',
        'fare_amount <= 0',
        'total_amount <= 0'
    ],
    'violating_rows': [
        invalid_duration,
        invalid_distance,
        invalid_fare,
        invalid_total
    ]
})
display(cleaning_audit)


In [ ]:
rows_before_base = len(df)

base_df = df[
    (df['trip_duration_min'] > 0) &
    (df['trip_distance'] > 0) &
    (df['fare_amount'] > 0) &
    (df['total_amount'] > 0)
].copy()

rows_after_base = len(base_df)

print('Rows before base cleaning:', rows_before_base)
print('Rows after base cleaning:', rows_after_base)
print('Rows removed:', rows_before_base - rows_after_base)
print('Removal ratio:', (rows_before_base - rows_after_base) / rows_before_base)


This step gives us a common cleaned dataset containing clearly valid completed trips.
Because the removal ratio is relatively small compared with the full dataset size, the cleaning remains conservative rather than overly restrictive.


### 3.3 Task-specific datasets

After establishing a common cleaned base dataset, we separate the data into two task-specific branches:

- `duration_df`: used for the main trip-duration regression task
- `tip_df`: used for supplementary payment/tipping analysis

This separation is important because post-trip monetary variables should not be used as features in the trip-duration prediction task.


In [ ]:
leakage_cols = [
    'fare_amount',
    'tip_amount',
    'total_amount',
    'extra',
    'mta_tax',
    'tolls_amount',
    'improvement_surcharge',
    'congestion_surcharge',
    'Airport_fee'
]

duration_df = base_df.drop(
    columns=[c for c in leakage_cols if c in base_df.columns]
).copy()

tip_df = base_df[
    base_df['tip_rate'].notna() &
    (base_df['tip_rate'] >= 0) &
    (base_df['tip_rate'] <= 1.0)
].copy()

print('duration_df shape:', duration_df.shape)
print('tip_df shape:', tip_df.shape)


In [ ]:
boundary_table = pd.DataFrame({
    'category': [
        'Main-model usable',
        'Leakage-prone / excluded from main model',
        'Tip-analysis specific'
    ],
    'examples': [
        'trip_distance, pickup_hour, pickup_dayofweek, is_weekend, passenger_count, location features',
        'fare_amount, tip_amount, total_amount, tolls_amount, congestion_surcharge, Airport_fee',
        'tip_rate, payment_type, fare_amount, tip_amount'
    ]
})
display(boundary_table)


### 3.4 Outlier handling

The taxi dataset contains heavy-tailed variables, especially `trip_duration_min` and `trip_distance`.
Because our main task is regression, extreme values may disproportionately affect model fitting, error metrics, and visualization readability.

We therefore first inspect summary statistics and upper quantiles of these variables, and then apply conservative upper-bound filters to remove only a very small number of clearly extreme observations.


In [ ]:
display(duration_df[['trip_duration_min', 'trip_distance']].describe())
display(duration_df[['trip_duration_min', 'trip_distance']].quantile([0.90, 0.95, 0.99, 0.995, 0.999]))


In [ ]:
MAX_DURATION = 180
MAX_DISTANCE = 100

duration_before = len(duration_df)
tip_before = len(tip_df)

duration_df = duration_df[
    (duration_df['trip_duration_min'] <= MAX_DURATION) &
    (duration_df['trip_distance'] <= MAX_DISTANCE)
].copy()

tip_df = tip_df[
    (tip_df['trip_duration_min'] <= MAX_DURATION) &
    (tip_df['trip_distance'] <= MAX_DISTANCE)
].copy()

duration_after = len(duration_df)
tip_after = len(tip_df)

print('duration_df after outlier filtering:', duration_df.shape)
print('tip_df after outlier filtering:', tip_df.shape)
print('Duration rows removed:', duration_before - duration_after)
print('Tip rows removed:', tip_before - tip_after)
print('Duration removal ratio:', (duration_before - duration_after) / duration_before)
print('Tip removal ratio:', (tip_before - tip_after) / tip_before)


Because only a very small fraction of observations is removed, these filters are conservative rather than aggressive.
Their purpose is mainly to reduce the influence of a tiny number of extreme trips on downstream modeling and visualization.


### 3.5 Location decoding and derived geographic features

Raw pickup and dropoff location IDs are useful but not very interpretable. To make the location signal easier to analyze and potentially more useful for prediction, we decode TLC location IDs using the taxi zone lookup table and create a small set of geographic features.

The five derived location features we prioritize for modeling are:
- `PU_Borough`: coarse origin geography
- `DO_Borough`: coarse destination geography
- `PU_service_zone`: whether the pickup starts in a core yellow zone, borough zone, or airport-related zone
- `same_borough`: whether pickup and dropoff occur within the same borough
- `airport_trip`: whether either endpoint is airport-related


In [ ]:
from pathlib import Path

lookup_candidates = [
    PROJECT_ROOT / 'data' / 'taxi_zone_lookup.csv',
    PROJECT_ROOT / 'taxi_zone_lookup.csv',
    PROJECT_ROOT / 'taxi_zone_lookup .csv'
]
lookup_path = next((p for p in lookup_candidates if p.exists()), None)

if lookup_path is None:
    print('Taxi zone lookup file not found; skipping location decoding.')
else:
    zone_lookup = pd.read_csv(lookup_path)

    pu_lookup = zone_lookup[['LocationID', 'Borough', 'Zone', 'service_zone']].rename(columns={
        'LocationID': 'PULocationID',
        'Borough': 'PU_Borough',
        'Zone': 'PU_Zone',
        'service_zone': 'PU_service_zone'
    })

    do_lookup = zone_lookup[['LocationID', 'Borough', 'Zone', 'service_zone']].rename(columns={
        'LocationID': 'DOLocationID',
        'Borough': 'DO_Borough',
        'Zone': 'DO_Zone',
        'service_zone': 'DO_service_zone'
    })

    decoded_cols = [
        'PU_Borough', 'PU_Zone', 'PU_service_zone',
        'DO_Borough', 'DO_Zone', 'DO_service_zone',
        'same_borough', 'airport_trip'
    ]

    for frame_name in ['duration_df', 'tip_df']:
        frame = locals()[frame_name].copy()
        frame = frame.drop(columns=[c for c in decoded_cols if c in frame.columns])
        frame = frame.merge(pu_lookup, on='PULocationID', how='left')
        frame = frame.merge(do_lookup, on='DOLocationID', how='left')

        frame['same_borough'] = (
            frame['PU_Borough'].notna() &
            frame['DO_Borough'].notna() &
            (frame['PU_Borough'] == frame['DO_Borough'])
        ).astype(int)

        airport_pickup = frame['PU_Zone'].str.contains('Airport', case=False, na=False) | frame['PU_Borough'].eq('EWR')
        airport_dropoff = frame['DO_Zone'].str.contains('Airport', case=False, na=False) | frame['DO_Borough'].eq('EWR')
        frame['airport_trip'] = (airport_pickup | airport_dropoff).astype(int)

        locals()[frame_name] = frame

    display(duration_df[['PULocationID', 'PU_Borough', 'PU_service_zone', 'DOLocationID', 'DO_Borough', 'same_borough', 'airport_trip']].head())
    print('Decoded duration_df shape:', duration_df.shape)
    print('Decoded tip_df shape:', tip_df.shape)


## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution of trip duration

We begin by examining the distribution of the target variable `trip_duration_min`. This helps us understand skewness, outliers, and whether additional filtering or transformation may be needed before modeling.


In [ ]:
duration_df['trip_duration_min'].hist(bins=100)
plt.xlabel('Trip duration (min)')
plt.ylabel('Count')
plt.title('Distribution of Trip Duration')
plt.show()


### 4.2 Relationship between trip duration and trip distance

One key concern raised in the project feedback was that trip duration may be heavily driven by trip distance alone. We therefore directly examine the relationship between these two variables before building more complex models.


In [ ]:
sample_eda = duration_df.sample(min(50000, len(duration_df)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(sample_eda['trip_distance'], sample_eda['trip_duration_min'], s=1)
plt.xlabel('Trip distance')
plt.ylabel('Trip duration (min)')
plt.title('Trip Duration vs Trip Distance')
plt.show()

display(duration_df[['trip_distance', 'trip_duration_min']].corr())


The scatter plot shows a clear positive relationship between trip distance and trip duration, indicating that distance is a major driver of travel time. However, the points are widely dispersed, especially for medium and long trips, suggesting that distance alone cannot fully explain variation in trip duration. This implies that additional factors such as pickup time, traffic conditions, and location may also contribute meaningful predictive information. The presence of several extreme observations further suggests that outliers or long-tail behavior should be considered during preprocessing.


### 4.3 Time-related patterns

We explore whether trip duration varies by pickup hour, day of week, and weekend status. These analyses help us understand whether time-related features add predictive value beyond trip distance.


In [ ]:
hourly_mean = duration_df.groupby('pickup_hour')['trip_duration_min'].mean()
hourly_mean.plot(kind='bar', figsize=(10, 4))
plt.title('Average Trip Duration by Pickup Hour')
plt.ylabel('Average Duration (min)')
plt.show()

dow_mean = duration_df.groupby('pickup_dayofweek')['trip_duration_min'].mean()
dow_mean.plot(kind='bar', figsize=(8, 4))
plt.title('Average Trip Duration by Pickup Day of Week')
plt.ylabel('Average Duration (min)')
plt.show()


### Analysis of time-related patterns

The two plots show that trip duration changes across both pickup hour and pickup day of week, which suggests that time-related features are useful for understanding and predicting trip duration.

For pickup hour, average trip duration is lowest during late-night and early-morning hours, and becomes higher during the daytime. This pattern is reasonable because traffic is usually lighter at night and heavier during the day, especially when commuting and other daily activities increase road usage. The rise in average duration during busier hours suggests that pickup time may capture information about congestion that is not explained by trip distance alone.

For pickup day of week, average trip duration appears to be higher on weekdays and lower on weekends. This also makes sense in practice. Weekday travel is more likely to be affected by work-related commuting and heavier traffic, while weekend travel may happen under less congested road conditions. This means that the day of week may also help explain variation in trip duration.

My view is that these plots show trip duration is influenced not only by how far the trip is, but also by when the trip happens. In other words, two trips with similar distances may still have different durations because traffic conditions and travel demand vary over time. Therefore, features such as `pickup_hour`, `pickup_dayofweek`, and `is_weekend` are likely to be useful in the prediction model.

One thing to note is that these plots are based on average values, so they do not show the full distribution within each group. Still, the overall patterns are clear enough to suggest that time-related variables provide additional predictive value beyond trip distance.


### 4.4 Location-related patterns

Pickup and dropoff locations are encoded as IDs, which are not directly interpretable as numerical variables. We therefore inspect the most common locations and, where possible, map them to more meaningful spatial categories such as borough or zone.


### Most frequent pickup locations

To better understand the spatial distribution of trips, we first examine the most frequent pickup locations in the dataset. Instead of only listing raw counts, we visualize the top 10 pickup location IDs. This helps show whether trips are evenly distributed across the city or concentrated in a small number of high-demand areas.

A highly uneven distribution would suggest that travel demand is spatially concentrated, meaning that location-related features may contain useful predictive information for trip duration.


In [ ]:
top_pu = duration_df['PULocationID'].value_counts().head(10)

plt.figure(figsize=(8, 4))
top_pu.sort_values().plot(kind='barh')
plt.title('Top 10 Pickup Locations')
plt.xlabel('Number of Trips')
plt.ylabel('PULocationID')
plt.show()


### Most frequent dropoff locations

We next examine the most frequent dropoff locations. Similar to pickup locations, this visualization helps us understand whether trips tend to end in a small number of popular destinations.

Comparing pickup and dropoff distributions may also reveal whether certain locations act as major travel hubs in both directions.


In [ ]:
top_do = duration_df['DOLocationID'].value_counts().head(10)

plt.figure(figsize=(8, 4))
top_do.sort_values().plot(kind='barh')
plt.title('Top 10 Dropoff Locations')
plt.xlabel('Number of Trips')
plt.ylabel('DOLocationID')
plt.show()


### Analysis of location-related patterns

The pickup and dropoff plots show that trip activity is not evenly distributed across all locations. Instead, a relatively small number of location IDs account for a large share of trips, indicating strong spatial concentration in the dataset.

Several IDs, such as 236, 237, and 161, appear frequently in both the pickup and dropoff rankings, suggesting that these locations may act as major travel hubs or highly active areas. At the same time, the two rankings are not identical, which implies that some locations may function more often as origins while others are more common as destinations.

These plots suggest that location-related variables may contain useful predictive information for trip duration, because trip activity is spatially concentrated rather than randomly distributed.

Although these plots do not directly show the relationship between location and trip duration, they suggest that location-related variables are not random background information. Trip activity is clearly concentrated in a relatively small number of pickup and dropoff locations, indicating strong spatial structure in the dataset. Since different locations may be associated with different traffic conditions, trip purposes, and travel patterns, this spatial structure may translate into meaningful predictive information for trip duration. Therefore, pickup and dropoff location variables should be considered as useful candidate features for duration prediction.


### 4.5 Supplementary payment and tipping analysis

Although payment and tipping variables are excluded from the main duration model to avoid leakage, we still analyze them separately as part of the supplementary statistical component of the project.


In [ ]:
tip_summary = tip_df.groupby('payment_type')['tip_rate'].agg(['count', 'mean', 'median']).sort_values('count', ascending=False)
display(tip_summary)


## 5. Feature Engineering for Modeling

For the main trip-duration prediction task, we only use variables that are available before or at the beginning of a trip. We exclude post-trip monetary variables such as fare amount, tip amount, and total amount to avoid data leakage.

Our modeling features include:
- trip distance
- passenger count
- pickup hour
- pickup day of week
- weekend indicator
- vendor / ratecode variables
- raw pickup/dropoff location IDs
- decoded location features such as borough, pickup service zone, same-borough indicator, and airport-trip indicator


In [ ]:
model_cols = [
    'trip_distance',
    'passenger_count',
    'pickup_hour',
    'pickup_dayofweek',
    'is_weekend',
    'VendorID',
    'RatecodeID',
    'PULocationID',
    'DOLocationID',
    'PU_Borough',
    'DO_Borough',
    'PU_service_zone',
    'same_borough',
    'airport_trip',
    'trip_duration_min'
]

model_df = duration_df[[c for c in model_cols if c in duration_df.columns]].dropna().copy()
display(model_df.head())
print('model_df shape:', model_df.shape)


## 6. Modeling

### 6.1 Train-test split and evaluation metrics

We use a train-test split for model evaluation and compare models using RMSE, MAE, and R². Our focus is not only predictive performance, but also understanding the incremental value of structured features beyond trip distance alone.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def eval_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}


### 6.2 Baseline 0: Distance-only Linear Regression

We first build a minimal baseline using only `trip_distance`. This directly addresses the concern that trip duration may be largely determined by distance alone.


In [ ]:
X = model_df[['trip_distance']]
y = model_df['trip_duration_min']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_dist = LinearRegression()
lr_dist.fit(X_train, y_train)
pred_dist = lr_dist.predict(X_test)

results = []
result_dist = eval_regression(y_test, pred_dist)
result_dist['Model'] = 'Distance-only Linear Regression'
results.append(result_dist)
display(pd.DataFrame([result_dist]))


### 6.3 Baseline 1: Multivariate Linear Regression

We then add time-related, passenger-related, and encoded location-related variables to test whether structured features improve prediction beyond trip distance alone.


In [ ]:
target_col = 'trip_duration_min'
feature_cols = [c for c in model_df.columns if c != target_col]

X = model_df[feature_cols]
y = model_df[target_col]

categorical_cols = [
    c for c in [
        'VendorID', 'RatecodeID', 'PULocationID', 'DOLocationID',
        'PU_Borough', 'DO_Borough', 'PU_service_zone'
    ] if c in X.columns
]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_model.fit(X_train, y_train)
pred_lr = lr_model.predict(X_test)

result_lr = eval_regression(y_test, pred_lr)
result_lr['Model'] = 'Multivariate Linear Regression'
results.append(result_lr)
display(pd.DataFrame([result_lr]))


### 6.4 Baseline 2: Ridge Regression

We use Ridge regression to handle possible multicollinearity and to stabilize the model when the number of encoded categorical features increases.


In [ ]:
ridge_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)
pred_ridge = ridge_model.predict(X_test)

result_ridge = eval_regression(y_test, pred_ridge)
result_ridge['Model'] = 'Ridge Regression'
results.append(result_ridge)
display(pd.DataFrame([result_ridge]))


### 6.5 Baseline 3: Lasso Regression

We use Lasso regression to examine whether some features can be shrunk toward zero, which may improve interpretability.


In [ ]:
lasso_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso(alpha=0.001, max_iter=5000))
])

lasso_model.fit(X_train, y_train)
pred_lasso = lasso_model.predict(X_test)

result_lasso = eval_regression(y_test, pred_lasso)
result_lasso['Model'] = 'Lasso Regression'
results.append(result_lasso)
display(pd.DataFrame([result_lasso]))


### 6.6 Advanced Models

After building interpretable linear and regularized baselines, we compare them with a more flexible model such as Random Forest.


In [ ]:
# To keep runtime manageable, sample a subset for Random Forest if needed
rf_sample = model_df.sample(min(200000, len(model_df)), random_state=42).copy()

X_rf = rf_sample[feature_cols]
y_rf = rf_sample[target_col]

categorical_cols_rf = [
    c for c in [
        'VendorID', 'RatecodeID', 'PULocationID', 'DOLocationID',
        'PU_Borough', 'DO_Borough', 'PU_service_zone'
    ] if c in X_rf.columns
]
numeric_cols_rf = [c for c in X_rf.columns if c not in categorical_cols_rf]

preprocessor_rf = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols_rf),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols_rf)
    ]
)

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)

rf_model = Pipeline([
    ('preprocessor', preprocessor_rf),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

rf_model.fit(X_train_rf, y_train_rf)
pred_rf = rf_model.predict(X_test_rf)

result_rf = eval_regression(y_test_rf, pred_rf)
result_rf['Model'] = 'Random Forest Regressor'
results.append(result_rf)
display(pd.DataFrame([result_rf]))


### 6.7 Model comparison

We summarize all model results in one table and compare whether additional complexity leads to meaningful improvement.


In [ ]:
results_df = pd.DataFrame(results)[['Model', 'RMSE', 'MAE', 'R2']]
display(results_df.sort_values('RMSE'))


## 7. Hypothesis Testing

We include a supplementary hypothesis testing component to study whether several group-level differences in the taxi data are statistically meaningful rather than due to random variation alone.

Our current hypotheses are:
- **Hypothesis 1:** mean `tip_rate` differs across the two most common payment types.
- **Hypothesis 2:** mean `trip_duration_min` differs between weekday and weekend trips.
- **Hypothesis 3:** mean `tip_rate` differs between daytime and nighttime trips.

Because the dataset is large and several variables are skewed, we use permutation tests for all three comparisons.


### 7.1 Test method

For each hypothesis, we compare two groups using the difference in sample means as the test statistic. We then repeatedly shuffle the pooled values across the two groups to generate a permutation distribution under the null hypothesis of no group effect.


In [ ]:
n_perm = 1000

def permutation_test_mean_diff(group1, group2, n_perm=1000, seed=42):
    group1 = np.asarray(group1)
    group2 = np.asarray(group2)

    observed_diff = group1.mean() - group2.mean()
    combined = np.concatenate([group1, group2])
    rng = np.random.default_rng(seed)
    perm_diffs = []

    for _ in range(n_perm):
        shuffled = rng.permutation(combined)
        perm_g1 = shuffled[:len(group1)]
        perm_g2 = shuffled[len(group1):]
        perm_diffs.append(perm_g1.mean() - perm_g2.mean())

    p_value = np.mean(np.abs(perm_diffs) >= abs(observed_diff))

    return {
        'observed_diff': observed_diff,
        'p_value': p_value,
        'perm_diffs': np.array(perm_diffs),
        'group1_mean': group1.mean(),
        'group2_mean': group2.mean(),
        'group1_n': len(group1),
        'group2_n': len(group2)
    }


### 7.2 Three planned hypothesis tests

Below, we implement all three current hypotheses using the same permutation-testing framework.


In [ ]:
hypothesis_results = []

# Hypothesis 1: payment type vs tip rate
common_types = tip_df['payment_type'].value_counts().head(2).index.tolist()
tip_sub = tip_df[tip_df['payment_type'].isin(common_types)].copy()
payment_g1 = tip_sub[tip_sub['payment_type'] == common_types[0]]['tip_rate'].values
payment_g2 = tip_sub[tip_sub['payment_type'] == common_types[1]]['tip_rate'].values
payment_result = permutation_test_mean_diff(payment_g1, payment_g2, n_perm=n_perm, seed=42)
payment_result['Hypothesis'] = 'Payment type: mean tip_rate differs'
payment_result['Group 1'] = f'payment_type = {common_types[0]}'
payment_result['Group 2'] = f'payment_type = {common_types[1]}'
hypothesis_results.append(payment_result)

# Hypothesis 2: weekday/weekend vs trip duration
weekday_g = duration_df[duration_df['is_weekend'] == 0]['trip_duration_min'].values
weekend_g = duration_df[duration_df['is_weekend'] == 1]['trip_duration_min'].values
weekend_result = permutation_test_mean_diff(weekday_g, weekend_g, n_perm=n_perm, seed=43)
weekend_result['Hypothesis'] = 'Weekday vs weekend: mean trip_duration_min differs'
weekend_result['Group 1'] = 'Weekday'
weekend_result['Group 2'] = 'Weekend'
hypothesis_results.append(weekend_result)

# Hypothesis 3: daytime/nighttime vs tip rate
tip_time_df = tip_df.copy()
tip_time_df['time_group'] = np.where(
    tip_time_df['pickup_hour'].between(6, 17),
    'Daytime',
    'Nighttime'
)
day_g = tip_time_df[tip_time_df['time_group'] == 'Daytime']['tip_rate'].values
night_g = tip_time_df[tip_time_df['time_group'] == 'Nighttime']['tip_rate'].values
daynight_result = permutation_test_mean_diff(day_g, night_g, n_perm=n_perm, seed=44)
daynight_result['Hypothesis'] = 'Daytime vs nighttime: mean tip_rate differs'
daynight_result['Group 1'] = 'Daytime (06:00-17:59)'
daynight_result['Group 2'] = 'Nighttime (18:00-05:59)'
hypothesis_results.append(daynight_result)

hypothesis_results_df = pd.DataFrame(hypothesis_results)[[
    'Hypothesis', 'Group 1', 'Group 2',
    'group1_mean', 'group2_mean', 'observed_diff', 'p_value', 'group1_n', 'group2_n'
]]
display(hypothesis_results_df)

print('Selected payment types for Hypothesis 1:', common_types)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Top row: more intuitive group-comparison visuals
axes[0, 0].bar(
    [f'payment_type = {common_types[0]}', f'payment_type = {common_types[1]}'],
    [payment_result['group1_mean'], payment_result['group2_mean']],
    color=['#4C78A8', '#72B7B2']
)
axes[0, 0].set_title('Mean tip rate by payment type')
axes[0, 0].set_ylabel('Mean tip_rate')

weekend_sample = duration_df.sample(min(50000, len(duration_df)), random_state=42).copy()
weekend_sample['week_group'] = np.where(weekend_sample['is_weekend'] == 1, 'Weekend', 'Weekday')
weekend_plot_data = [
    weekend_sample.loc[weekend_sample['week_group'] == 'Weekday', 'trip_duration_min'],
    weekend_sample.loc[weekend_sample['week_group'] == 'Weekend', 'trip_duration_min']
]
axes[0, 1].boxplot(weekend_plot_data, labels=['Weekday', 'Weekend'], showfliers=False)
axes[0, 1].set_title('Trip duration by weekday/weekend')
axes[0, 1].set_ylabel('trip_duration_min')

axes[0, 2].bar(
    ['Daytime', 'Nighttime'],
    [daynight_result['group1_mean'], daynight_result['group2_mean']],
    color=['#4C78A8', '#72B7B2']
)
axes[0, 2].set_title('Mean tip rate by time of day')
axes[0, 2].set_ylabel('Mean tip_rate')

# Bottom row: permutation distributions kept as statistical support
plot_inputs = [
    ('Payment type permutation test', payment_result),
    ('Weekday/weekend permutation test', weekend_result),
    ('Day/night permutation test', daynight_result)
]

for ax, (title, result) in zip(axes[1], plot_inputs):
    ax.hist(result['perm_diffs'], bins=40, color='#BDBDBD', edgecolor='white')
    ax.axvline(0, color='black', linewidth=1, label='Null center (0 difference)')
    ax.axvline(
        result['observed_diff'],
        color='crimson',
        linestyle='--',
        linewidth=2,
        label=f"Observed difference = {result['observed_diff']:.3f}"
    )
    ax.set_title(title)
    ax.set_xlabel('Permuted mean difference')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()


### 7.3 Interpretation

All three permutation tests return p-values that are effectively 0 in our simulation output, which means that under random reassignment of group labels, we almost never observe mean differences as large as those in the real data. This provides strong evidence against the corresponding null hypotheses for all three comparisons.

For **Hypothesis 1**, the two most common payment types have a large difference in mean tip rate: approximately `0.248` versus `0.018`, with an observed difference of about `0.230`. This is both statistically significant and substantively large, so payment type appears strongly associated with tipping behavior in our filtered sample.

For **Hypothesis 2**, weekday trips have a higher average trip duration than weekend trips in the filtered duration dataset, about `17.78` minutes versus `15.48` minutes, with an observed difference of about `2.30` minutes. Because the p-value is effectively 0, we reject the null hypothesis of equal mean trip duration across weekday and weekend groups. This result suggests that temporal traffic or travel-demand patterns differ meaningfully across the week.

For **Hypothesis 3**, the daytime mean tip rate is about `0.164`, compared with about `0.153` at night, for an observed difference of about `0.010`. This difference is statistically significant given the very large sample size, but it is much smaller in magnitude than the payment-type result. We therefore interpret this finding more cautiously: the data suggests a real day/night tipping difference, but the practical effect size appears modest.

Overall, these results illustrate why statistical significance and practical importance should be discussed separately. In a dataset with millions of observations, even a small mean difference can produce an extremely small p-value. For that reason, our final interpretation emphasizes both the p-values and the actual size of the observed group differences.


## 8. Challenges and Limitations

This project faces several practical and methodological challenges:

1. **Large dataset size and runtime constraints:** even after cleaning, the dataset still contains millions of observations. This affects memory use, visualization readability, and model-training time. In practice, we had to use conservative filtering and subsampling for some analyses, especially the Random Forest model and some visualizations.

2. **Outliers and heavy-tailed variables:** trip duration and trip distance are both skewed and contain extreme observations. Although we applied conservative upper-bound filters, the choice of thresholds can still influence both model fit and interpretation.

3. **Location encoding and interpretability:** pickup and dropoff locations are recorded as IDs rather than naturally interpretable spatial variables. We improved this by decoding locations and adding borough- and airport-related features, but location effects are still only partially captured and remain difficult to interpret fully.

4. **Data leakage concerns:** for the main prediction task, post-trip monetary variables such as fare amount, tip amount, and total amount must be excluded. Our modeling pipeline is intentionally leakage-aware, but this also means the prediction task is more realistic and therefore harder.

5. **Model complexity versus interpretability:** trip distance alone already explains a substantial share of variation in trip duration. This means that more complex models must be justified by clear performance gains rather than added complexity alone. In our results, Random Forest improves predictive accuracy, but the linear and regularized models remain easier to explain.

6. **Limits of hypothesis testing in observational data:** our permutation tests show statistically significant differences across groups, but these should not be interpreted as causal effects. Group differences may also reflect omitted variables, selection effects, or broader travel-pattern differences that are not fully controlled in our simple group comparisons.

7. **Scope limitation:** this project uses one month of NYC yellow taxi data and a limited set of structured variables. As a result, the findings may not generalize perfectly across seasons, across all taxi services, or under changing traffic and weather conditions.


## 9. Conclusion and Future Work

In this project, we built a leakage-aware regression pipeline for predicting NYC yellow taxi trip duration using structured trip-level data. We began with a simple baseline and then evaluated whether additional temporal, passenger, and location-related features improve prediction beyond trip distance alone.

Our modeling results show that `trip_distance` is a strong baseline predictor, but richer structured features provide meaningful improvement. The distance-only linear regression achieved an RMSE of about `9.55` and an R^2 of about `0.591`, while the multivariate linear model improved to an RMSE of about `7.91` and an R^2 of about `0.719`. Ridge and Lasso produced similar results, and the best predictive performance came from the Random Forest regressor, with an RMSE of about `6.70`, an MAE of about `3.86`, and an R^2 of about `0.796`. Overall, this suggests that nonlinear structure and interaction effects matter, but interpretable linear baselines still provide a useful benchmark.

As a supplementary statistical component, we also tested several hypotheses about tipping behavior and trip timing. All three permutation tests returned p-values that were effectively 0 in our simulation output, indicating strong evidence of group-level differences in the observed data. The largest practical difference appeared in mean tip rate across payment types, while the daytime versus nighttime tip-rate difference was statistically significant but comparatively small in magnitude. This reinforces an important project takeaway: in a very large dataset, statistical significance should always be interpreted together with effect size.

Taken together, our results show that NYC taxi trip duration can be predicted reasonably well from information available at or near trip start, and that travel and tipping behavior both vary systematically across meaningful subgroups. The project therefore combines predictive modeling with interpretable behavioral analysis in a realistic, no-leakage setting.

Future work could extend this project in several directions:
- incorporate taxi zone information more systematically and engineer richer geographic features,
- expand the analysis to multiple months or seasons to test temporal stability,
- add external variables such as weather, traffic, or special-event indicators,
- tune nonlinear models more carefully and compare additional methods such as gradient boosting,
- and explore related prediction tasks, such as fare or cost prediction, under a carefully designed no-leakage setup.
